In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RESULTS = Path("../results")

CONF = RESULTS / "week12" / "confirmatory"
HARM = RESULTS / "week12" / "harmonized_n100"
OUT = RESULTS / "week12" / "harmonized_analysis"

OUT.mkdir(parents=True, exist_ok=True)

In [2]:
confirmatory = pd.read_csv(
    CONF / "primary_power_runs.csv",
    keep_default_na=False,
    na_values=[""]
)

confirmatory = confirmatory[
    confirmatory["N"].isin([50, 200])
].copy()

confirmatory["source_status"] = "confirmatory"

In [3]:
n100 = pd.read_csv(
    HARM / "harmonized_n100_power_runs.csv",
    keep_default_na=False,
    na_values=[""]
)

assert set(n100["N"]) == {100}

n100["source_status"] = "posthoc_harmonized_n100"

In [4]:
runs = pd.concat(
    [confirmatory, n100],
    ignore_index=True
)

In [5]:
print(runs.shape)
print(runs["N"].value_counts().sort_index())

(37000, 19)
N
50     12000
100    15000
200    10000
Name: count, dtype: int64


In [6]:
assert len(runs) == 37000
assert runs["run_id"].is_unique
assert runs["reject"].isin([0, 1]).all()

In [7]:
runs["cluster_id"] = (
    runs["scenario"].astype(str)
    + "|R"
    + runs["rep"].astype(str)
)

In [8]:
runs["sample_cluster"] = (
    runs["scenario"].astype(str)
    + "|N"
    + runs["N"].astype(str)
    + "|R"
    + runs["rep"].astype(str)
)

In [9]:
runs.to_csv(
    OUT / "harmonized_repetition_level_power.csv",
    index=False
)

In [10]:
formula = (
    "reject ~ "
    "C(scenario) + "
    "C(severity) + "
    "C(N) + "
    "C(test) + "
    "C(scenario):C(test) + "
    "C(severity):C(test) + "
    "C(N):C(test)"
)

In [11]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

gee = smf.gee(
    formula=formula,
    groups="cluster_id",
    data=runs,
    family=sm.families.Binomial(),
    cov_struct=sm.cov_struct.Exchangeable()
)

gee_result = gee.fit()

print(gee_result.summary())

                               GEE Regression Results                              
Dep. Variable:                      reject   No. Observations:                37000
Model:                                 GEE   No. clusters:                     1000
Method:                        Generalized   Min. cluster size:                  34
                      Estimating Equations   Max. cluster size:                  43
Family:                           Binomial   Mean cluster size:                37.0
Dependence structure:         Exchangeable   Num. iterations:                    15
Date:                     Fri, 20 Nov 2026   Scale:                           1.000
Covariance type:                    robust   Time:                         13:05:21
                                                                         coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------------

#### Joint Wald test for N × Test Configuration: primary clustering


In [12]:
# Identify N × Test Configuration interaction coefficients
interaction_terms_main = [
    name
    for name in gee_result.params.index
    if "C(N)" in name and ":C(test)" in name
]

print("N × Test interaction terms:")
for term in interaction_terms_main:
    print(" -", term)

# Joint null hypothesis:
# all N × Test Configuration interaction coefficients = 0
param_names_main = list(gee_result.params.index)

R_main = np.zeros(
    (len(interaction_terms_main), len(param_names_main))
)

for i, term in enumerate(interaction_terms_main):
    j = param_names_main.index(term)
    R_main[i, j] = 1

wald_n_test_main = gee_result.wald_test(
    R_main,
    scalar=True
)

print("\nPrimary clustering: Scenario × repetition")
print("Wald chi-square:", float(wald_n_test_main.statistic))
print("df:", len(interaction_terms_main))
print("p-value:", float(wald_n_test_main.pvalue))

N × Test interaction terms:
 - C(N)[T.100]:C(test)[T.C2ST-RBF]
 - C(N)[T.200]:C(test)[T.C2ST-RBF]
 - C(N)[T.100]:C(test)[T.Energy]
 - C(N)[T.200]:C(test)[T.Energy]
 - C(N)[T.100]:C(test)[T.MMD]
 - C(N)[T.200]:C(test)[T.MMD]
 - C(N)[T.100]:C(test)[T.Sinkhorn]
 - C(N)[T.200]:C(test)[T.Sinkhorn]

Primary clustering: Scenario × repetition
Wald chi-square: 126.62662799668044
df: 8
p-value: 1.4139910468441387e-23


#### GEE sensitivity analysis: alternative clustering by Scenario x N x repetition
As a clustering sensitivity analysis, the logistic GEE was refitted using scenario x N x repetition as the cluster definition instead of scenario x repetition. The fixed-effects specification, binomial family, exchangeable working correlation structure, and robust covariance estimator were kept unchanged. This analysis evaluates whether the main inferential conclusion depends materially on the clustering specification.

In [13]:

gee_sensitivity = smf.gee(
    formula=formula,
    groups="sample_cluster",
    data=runs,
    family=sm.families.Binomial(),
    cov_struct=sm.cov_struct.Exchangeable()
)

gee_sensitivity_result = gee_sensitivity.fit()

print(gee_sensitivity_result.summary())

                               GEE Regression Results                              
Dep. Variable:                      reject   No. Observations:                37000
Model:                                 GEE   No. clusters:                     3000
Method:                        Generalized   Min. cluster size:                   5
                      Estimating Equations   Max. cluster size:                  15
Family:                           Binomial   Mean cluster size:                12.3
Dependence structure:         Exchangeable   Num. iterations:                    12
Date:                     Fri, 20 Nov 2026   Scale:                           1.000
Covariance type:                    robust   Time:                         13:05:37
                                                                         coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------------

#### Joint Wald test for the N x Test Configuration interaction
The joint Wald test evaluates whether the full set of N x Test Configuration interaction coefficients is simultaneously equal to zero under the alternative clustering specification. This is the sensitivity-analysis counterpart of the main global interaction test.

In [14]:
# Identify all N x Test Configuration interaction coefficients
interaction_terms = [
    name
    for name in gee_sensitivity_result.params.index
    if "C(N)" in name and ":C(test)" in name
]

print("N × Test interaction terms:")
for term in interaction_terms:
    print(" -", term)

# Build restriction matrix R such that H0: all N x Test coefficients = 0
param_names = list(gee_sensitivity_result.params.index)

R = np.zeros(
    (len(interaction_terms), len(param_names))
)

for i, term in enumerate(interaction_terms):
    j = param_names.index(term)
    R[i, j] = 1

wald_n_test_sensitivity = gee_sensitivity_result.wald_test(
    R,
    scalar=True
)

print("\nAlternative clustering: Scenario × N × repetition")
print("Wald chi-square:", float(wald_n_test_sensitivity.statistic))
print("df:", len(interaction_terms))
print("p-value:", float(wald_n_test_sensitivity.pvalue))

N × Test interaction terms:
 - C(N)[T.100]:C(test)[T.C2ST-RBF]
 - C(N)[T.200]:C(test)[T.C2ST-RBF]
 - C(N)[T.100]:C(test)[T.Energy]
 - C(N)[T.200]:C(test)[T.Energy]
 - C(N)[T.100]:C(test)[T.MMD]
 - C(N)[T.200]:C(test)[T.MMD]
 - C(N)[T.100]:C(test)[T.Sinkhorn]
 - C(N)[T.200]:C(test)[T.Sinkhorn]

Alternative clustering: Scenario × N × repetition
Wald chi-square: 128.81663695840456
df: 8
p-value: 4.975972028362472e-24


#### Clustering sensitivity result
The primary GEE clustered observations by Scenario × repetition, whereas the sensitivity specification clustered by Scenario x N x repetition.

The joint N x Test Configuration interaction remained strongly supported under the alternative clustering specification. The magnitude of the Wald statistic was very similar to that obtained with the primary clustering definition, indicating that the main conclusion was not materially dependent on whether
sample size was included in the cluster identifier.

This analysis is post-preregistration and is interpreted as a sensitivity analysis rather than as primary confirmatory evidence.

In [15]:
from statsmodels.stats.contingency_tables import mcnemar

def paired_test(df, test_a, test_b):

    a = df[df["test"] == test_a][
        ["scenario", "severity", "N", "rep", "reject"]
    ].rename(columns={"reject": "a"})

    b = df[df["test"] == test_b][
        ["scenario", "severity", "N", "rep", "reject"]
    ].rename(columns={"reject": "b"})

    paired = a.merge(
        b,
        on=["scenario", "severity", "N", "rep"],
        how="inner"
    )

    table = pd.crosstab(
        paired["a"],
        paired["b"]
    ).reindex(
        index=[0,1],
        columns=[0,1],
        fill_value=0
    )

    result = mcnemar(
        table,
        exact=False,
        correction=True
    )

    return paired, table, result

In [16]:
paired, table, result = paired_test(
    runs,
    "C2ST-Linear",
    "MMD"
)

print(table)
print(result)

b    0     1
a           
0  792  3104
1   19  2485
pvalue      0.0
statistic   3045.4870317002883
